In [1]:
import math

def decimal_to_bin (decimal_value, INT_BITS=5, FRAC_BITS=10):

    # check if i will need to 2's complement the answer at the end
    twos_comp = 1 if decimal_value < 0 else 0

    # extract the integer and fraction parts to convert each one alone
    abs_value = abs(decimal_value)
    integer_part = int(abs_value)
    fraction_part = abs_value - integer_part


    # assume all are zeros in the start
    integer_bits  =  [0]  *  INT_BITS           # 5 bits for integer
    fraction_bits =  [0]  * FRAC_BITS           # 10 bits for fraction


    ########### Generating Integer Bits #############
    # i start from 4 to 0
    for i in reversed(range(INT_BITS)):         # moving from MSB(2^4) weight to LSB(2^0)
        weight = pow(2,i)                # current index weight
        remaining = integer_part - weight

        if remaining >= 0:
            integer_bits[INT_BITS-1-i] = 1        # MSB is index [0] in the array / LSB is [4]
            integer_part = remaining
            #print(integer_bits)


    ############ Generating Fraction Bits ##############
    # i start from 0 to 9
    for i in range(FRAC_BITS):                  # moving from MSB(2^-1) to LSB(2^-10)
        weight = pow(2,-(i+1))           # weights starts from -1 no 0 in fraction
        remaining = fraction_part - weight

        if remaining >= 0:
            fraction_bits[i] = 1       # MSB is index [0] in the array / LSB is [9]
            fraction_part = remaining
            #print(fraction_bits)


    ########### 2's Complementing #############
    all_bits = integer_bits + fraction_bits

    if twos_comp == 1:
        found_first_one = False
        for i in reversed(range(len(all_bits))):

            if found_first_one:
                all_bits[i] = 1 - all_bits[i]   # Invert bits after finding the first 1

            elif all_bits[i] == 1:
                found_first_one = True          # Found the first 1, keep it and set flag


    #print(f"Unsigned Integer Bits: {integer_bits}")
    #print(f"Unsigned Fraction Bits: {fraction_bits}")
    # We are DONE
    return all_bits

def decimal_to_lns(decimal_value, INT_BITS=5, FRAC_BITS=10):

    abs_value = abs(decimal_value)
    log_value = math.log2(abs_value)
    exponent = decimal_to_bin(log_value, INT_BITS, FRAC_BITS)

    logarithmic_bits = exponent
   # logarithmic_bits.append(sign , exponent)

    formated_output = ''.join(map(str,logarithmic_bits))

    #print(f"Sign Bit: {sign}")
    return formated_output

def phi_rom(INT_BITS=5, FRAC_BITS=10):
    """
    توليد بيانات ROM للـ phi_plus بناءً على عدد البتات للجزء الصحيح والكسر
    """
    WIDTH     = INT_BITS + FRAC_BITS   # +1 للـ sign bit
    DATA_BITS = WIDTH - 1                  # عرض البيانات
    entries   = 2 ** DATA_BITS             # عدد الـ entries
    step      = 2 ** (-FRAC_BITS)          # دقة الكسر

    rom_phi_plus = [0] * (entries + 1)

    for address in range(entries + 1):
        # d = ly - lx
        d = -2 ** (INT_BITS-1) + (address * step)
        two_d = 2 ** d
        # phi_plus = log(1 + 2^d) في الـ log-domain
        rom_phi_plus[entries - address] = decimal_to_lns(1 + two_d, INT_BITS, FRAC_BITS)

    return rom_phi_plus, WIDTH, DATA_BITS, entries


def export_rom_data(INT_BITS=5, FRAC_BITS=10, threshold=1000):
    """
    Generate ROM data and export it to text file for use with SystemVerilog $readmemb
    Also detect long runs of repeated values
    """
    rom_phi_plus, WIDTH, DATA_BITS, entries = phi_rom(INT_BITS, FRAC_BITS)

    filename = f"rom_phi_plus.txt"
    with open(filename, 'w') as f:
        for value in rom_phi_plus:
            if not isinstance(value, str):
                value = ''.join(map(str, value))
            f.write(f"{value}\n")

    print(f"Successfully exported {len(rom_phi_plus)} entries ({DATA_BITS} bits wide) to {filename}")

    # Detect long runs of repeated values
    prev_val = None
    run_length = 0
    run_start = 0
    for addr, val in enumerate(rom_phi_plus):
        if val == prev_val:
            run_length += 1
        else:
            if run_length >= threshold:
                print(f"Value {prev_val} repeated {run_length} times starting at address {run_start}")
            run_length = 1
            run_start = addr
            prev_val = val

    # Check last run
    if run_length >= threshold:
        print(f"Value {prev_val} repeated {run_length} times starting at address {run_start}")


# مثال تشغيل
export_rom_data(INT_BITS=6, FRAC_BITS=11, threshold=100 )

Successfully exported 65537 entries (16 bits wide) to rom_phi_plus.txt
Value 00000000000011101 repeated 101 times starting at address 13547
Value 00000000000011100 repeated 104 times starting at address 13648
Value 00000000000011011 repeated 108 times starting at address 13752
Value 00000000000011010 repeated 112 times starting at address 13860
Value 00000000000011001 repeated 116 times starting at address 13972
Value 00000000000011000 repeated 121 times starting at address 14088
Value 00000000000010111 repeated 127 times starting at address 14209
Value 00000000000010110 repeated 131 times starting at address 14336
Value 00000000000010101 repeated 138 times starting at address 14467
Value 00000000000010100 repeated 145 times starting at address 14605
Value 00000000000010011 repeated 152 times starting at address 14750
Value 00000000000010010 repeated 160 times starting at address 14902
Value 00000000000010001 repeated 170 times starting at address 15062
Value 00000000000010000 repeated

In [ ]:
with open("rom_phi_plus.txt", "r") as f_in, open("rom_phi_plus1.txt", "w") as f_out:
    for line in f_in:
        bin_str = line.strip()
        if bin_str:
            hex_str = format(int(bin_str, 2), "X")  # تحويل من binary إلى hex
            f_out.write(hex_str + "\n")

In [ ]:
def to_intel_hex(input_file, output_file, word_width=17, depth=32768):
    """
    يحوّل ملف فيه قيم Hex عادية (سطر لكل كلمة) إلى ملف Intel HEX.
    - input_file: اسم الملف الأصلي (قيم hex عادية).
    - output_file: اسم ملف Intel HEX الناتج.
    - word_width: عرض الكلمة بالبت (هنا 17).
    - depth: عدد الكلمات (هنا 32768).
    """

    def checksum(byte_list):
        s = sum(byte_list)
        return ((~s + 1) & 0xFF)

    with open(input_file, "r") as f:
        words = [line.strip() for line in f if line.strip()]

    with open(output_file, "w") as out:
        addr = 0
        for w in words:
            val = int(w, 16)
            # حوّل الكلمة إلى 3 بايت (LSB أولاً)
            b0 = val & 0xFF
            b1 = (val >> 8) & 0xFF
            b2 = (val >> 16) & 0xFF
            data = [b0, b1, b2]

            # record: length=03, address=addr, type=00
            record = [0x03, (addr >> 8) & 0xFF, addr & 0xFF, 0x00] + data
            cs = checksum(record)
            line = ":" + "".join(f"{x:02X}" for x in record) + f"{cs:02X}"
            out.write(line + "\n")

            addr += 3  # كل كلمة 3 بايت

        # نهاية الملف
        out.write(":00000001FF\n")


# مثال استخدام:
to_intel_hex("rom_phi_plus1.txt", "rom_phi_plus.hex", word_width=17, depth=32768)